In [0]:
%run ../source_to_bronze/utils

In [0]:
silver_path = "/Volumes/sample/default/assignment_volume/silver/Employee_info/dim_employee"

employee_gold_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

In [0]:
display(employee_gold_df)

In [0]:
from pyspark.sql.functions import *

In [0]:
department_salary_df = (
    employee_gold_df
    .groupBy("department")
    .agg(
        sum("salary").alias("total_salary")
    )
    .orderBy(
        col("total_salary").desc()
    )
)

In [0]:
display(department_salary_df)

In [0]:
employee_count_df = (
    employee_gold_df
    .groupBy(
        "department",
        "country"
    )
    .agg(
        count("employee_i_d").alias("employee_count")
    )
)

In [0]:
display(employee_count_df)

In [0]:
department_country_df = (
    employee_gold_df
    .select(
        "department",
        "country"
    )
    .distinct()
)

In [0]:
display(department_country_df)

In [0]:
average_age_df = (
    employee_gold_df
    .groupBy("department")
    .agg(
        avg("age").alias("average_age")
    )
)

In [0]:
display(average_age_df)

In [0]:
gold_df = (
    employee_gold_df
    .groupBy(
        "department",
        "country"
    )
    .agg(
        sum("salary").alias("total_salary"),
        count("employee_i_d").alias("employee_count"),
        avg("age").alias("average_age")
    )
)

In [0]:
gold_df = gold_df.orderBy(
    col("total_salary").desc()
)

In [0]:
display(gold_df)

In [0]:
gold_df = gold_df.withColumn(
    "at_load_date",
    current_date()
)

In [0]:
display(gold_df)

In [0]:
gold_path = "/Volumes/sample/default/assignment_volume/gold/employee/fact_employee"

In [0]:
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "replaceWhere",
        f"at_load_date = '{spark.sql('SELECT current_date()').collect()[0][0]}'"
    )
    .save(gold_path)
)

In [0]:
final_gold_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)

display(final_gold_df)

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/sample/default/assignment_volume/gold/employee/fact_employee`